# Connect4

-----general introduction to the work----------

In [ ]:
import random
import math 
import csv
import os
import pickle
import pandas as pd
import numpy as np

## Game Logic

In [ ]:
ROWS = 6
COLS = 7

def create_board():
  
    return [[0] * COLS for _ in range(ROWS)]

def print_board(board):
    
    for row in board:
        print(" ".join("X" if cell == 1 else "O" if cell == 2 else "-" for cell in row))

def is_valid_move(board, col):
    
    return board[0][col] == 0

def get_next_open_row(board, col):
    
    for row in range(ROWS - 1, -1, -1):  
        if board[row][col] == 0:
            return row
    print("[ERROR] No row available")
    return None  

def drop_piece(board, row, col, piece):
    
    board[row][col] = piece

def check_winner(board, piece):
    
    
    for r in range(ROWS):
        for c in range(COLS - 3):
            if all(board[r][c + i] == piece for i in range(4)):
                return True

    
    for r in range(ROWS - 3):
        for c in range(COLS):
            if all(board[r + i][c] == piece for i in range(4)):
                return True

    
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            if all(board[r + i][c + i] == piece for i in range(4)):
                return True

    
    for r in range(3, ROWS):
        for c in range(COLS - 3):
            if all(board[r - i][c + i] == piece for i in range(4)):
                return True

    return False

def get_valid_moves(board):

    return [column for column in range(COLS) if is_valid_move(board, column)]

def board_to_string(board):
    return ''.join([str(cell) for row in board for cell in row])

 ## Random Algorithm


In [ ]:


def get_random_move(**kwargs):
    board = kwargs.get("board")
    
    return random.choice(get_valid_moves(board))

## Monte Carlo Tree Search Algorithm

In [ ]:
class MCTSNode:
    def __init__(self, board, player, parent=None, move=None):
        self.board = [row[:] for row in board]  # Copy of the board at this node
        self.player = player              # The player who made the move leading here
        self.parent = parent              # Parent node
        self.move = move                  # Move that was made to reach this node (column)
        self.children = []                # Child nodes
        self.visits = 0                   # N: number of times node was visited
        self.wins = 0                     # W: number of wins from this node
        self.untried_moves = get_valid_moves(board)  

    def ucb1(self, total_simulations, c=1.41):
        if self.visits == 0:
            return float('inf')  
        win_rate = self.wins / self.visits
        return win_rate + c * math.sqrt(math.log(total_simulations) / self.visits)

    def is_fully_expanded(self):
        return len(self.untried_moves) == 0 or check_winner(self.board,1) or check_winner(self.board,2)

def MCTS(**kwargs):
    root = kwargs.get("root")
    iterations = kwargs.get("iterations", 500)
    bestchild = kwargs.get("bestchild", bestchild_default)
    backprograming = kwargs.get("backprograming", backprograming_default)
    c = kwargs.get("c", 2.0)

    for _ in range(iterations):
        node = selection(root,c=c)
        if not node.is_fully_expanded():
            child = expansion(node)
            result = simulation(child)
            backprograming(child, result)
        else:
            result = node.player if check_winner(node.board,  node.player) else 0
            backprograming(node, result)

    chosen = bestchild(root)
    if chosen is None:
        print("[MCTS DEBUG] No child chosen — fallback to random move.")
        return random.choice(get_valid_moves(root.board))

    return chosen.move

def selection(node: MCTSNode, c=1.41):
    while True:
    
        if not node.is_fully_expanded():
            return node

        if not node.children:
            return node

        node = max(node.children, key=lambda child: child.ucb1(node.visits,c))

def expansion(node: MCTSNode) ->MCTSNode:
    
    move=random.choice(node.untried_moves)
    node.untried_moves.remove(move)
    
    row=get_next_open_row(node.board,move)
    new_board=[row[:] for row in node.board]
    
    drop_piece(new_board,row,move,3-node.player)


    expanded_node=MCTSNode(board=new_board,player=3-node.player,parent=node,move=move)

    node.children.append(expanded_node)

    return expanded_node

def get_smart_move(board, player):
  
    for col in get_valid_moves(board):
        row = get_next_open_row(board, col)
        temp_board = [r[:] for r in board]
        drop_piece(temp_board, row, col, player)
        if check_winner(temp_board, player):
            return col

   
    opponent = 3 - player
    for col in get_valid_moves(board):
        row = get_next_open_row(board, col)
        temp_board = [r[:] for r in board]
        drop_piece(temp_board, row, col, opponent)
        if check_winner(temp_board, opponent):
            return col

   
    return random.choice(get_valid_moves(board))

def simulation(node: MCTSNode) ->int:
    board=[row[:] for row in node.board]
    player= 3 - node.player
    while get_valid_moves(board):
    
        move=get_smart_move(board=board,player=player)
        row=get_next_open_row(board,move)

        drop_piece(board,row,move,player)
        
        if check_winner(board,player):
            return player
        player=3-player
    return 0

def backprograming_default(node : MCTSNode,result):
    
    while node is not None:
        node.visits+=1

        if result== node.player: 
            
            node.wins+=1

        elif result==0:
            node.wins+=0.5
        node=node.parent

def backprograming_greddy(node : MCTSNode,result):

    while node is not None:
        node.visits+=1

        if result== node.player: 
            
            node.wins+=1

        elif result==0:
            node.wins+=0.15
        node=node.parent

def bestchild_default(node: MCTSNode):
    if not node.children:
        print("[BESTCHILD DEBUG] No children in node.")
        return None

    visited_children = [child for child in node.children if child.visits > 0]
    if not visited_children:
        print("[BESTCHILD DEBUG] All children have 0 visits.")
        return None

    best = max(visited_children, key=lambda child: child.wins / child.visits)

    return best

def bestchild_higherVisits(node : MCTSNode):
    if not node.children:
        print("[BESTCHILD DEBUG] No children in node.")
        return None

    visited_children = [child for child in node.children if child.visits > 0]
    if not visited_children:
        print("[BESTCHILD DEBUG] All children have 0 visits.")
        return None

    best = max(visited_children, key=lambda child: child.visits)

    return best

def update_root(root,board,col,turn):   

    matching_child = next((child for child in root.children if child.move == col), None)
    if matching_child:
        root = matching_child
        root.parent = None
    else:
       
        root = MCTSNode(board=board, player= turn)
    return root 


### Save moves data

In [ ]:
def save_to_csv(data, filename="mcts_moves.csv"):
    fieldnames = [f"s{i}" for i in range(42)] + ["move"]

    write_header = not os.path.exists(filename)

    with open(filename, mode='a', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)

        if write_header:
            writer.writeheader()

        for entry in data:
            state_str = entry['state'] 
            move = entry['move']

            row = {f"s{i}": int(state_str[i]) for i in range(42)}
            row["move"] = move

            writer.writerow(row)

## Player vs Player/Ai

In [ ]:
def play_game(mode="pvc", ai=None):
    board = create_board()
    game_over = False
    turn = 1  
    root=None
    if ai==MCTS:
        root=MCTSNode(board,1)
    
    print_board(board)

    while not game_over:
        
        if mode == "pvp" or (mode == "pvc" and turn == 1):
            
            col = None
            while col is None:
                try:
                    print(f"Player {turn} ({'X' if turn == 1 else 'O'}), choose column (0-{COLS-1}): ")
                    col = int(input())
                    print(f"Player {turn} selectd cloumn {col}")
                    if col not in get_valid_moves(board):
                        print("Invalid move! Try again.")
                        col = None
                except ValueError:
                    print("Invalid input! Enter a number.")
        else:
            if ai:
                col = ai(root=root,board=board)
            
            else:
                raise ValueError("AI strategy not provided for computer player.")
            print(f"Computer {turn} (O) chooses column {col}")
        
        row = get_next_open_row(board, col)
        drop_piece(board, row, col, turn)
        
        print_board(board)

        
        if root:
           root=update_root(root,board,col,turn)
 
        if check_winner(board, turn):
            print(f"Player {turn} ({'X' if turn == 1 else 'O'}) wins!")
            return turn
        elif not get_valid_moves(board):
            print("It's a draw!")
            return 0

        turn = 3 - turn  

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def play_game_jupyter(mode="pvc", ai=None):
    board = create_board()
    game_over = False
    turn = 1
    root = None

    # Variáveis para mostrar a última jogada corretamente
    last_move = None
    last_symbol = None
    last_player = None

    if ai == MCTS:
        root = MCTSNode(board, 1)

    output = widgets.Output()
    display(output)

    def render():
        with output:
            clear_output(wait=True)
            print_board(board)

            # Mostrar última jogada
            if last_move is not None and last_symbol is not None and last_player is not None:
                if mode == "pvc" and last_player == 2:
                    jogador = "Computador"
                else:
                    jogador = "Jogador"
                print(f"Última jogada: {jogador} ({last_symbol}) jogou na coluna {last_move}")
            
           
            # Mostrar quem vai jogar a seguir
            proximo_simbolo = 'X' if turn == 1 else 'O'
            proximo_jogador = "Jogador" if (mode == "pvp" or turn == 1) else "Computador"
            print(f"Próximo a jogar: {proximo_jogador} ({proximo_simbolo})")

    def on_button_click(b):
        nonlocal turn, board, root, game_over
        nonlocal last_move, last_symbol, last_player

        if game_over:
            return

        col = int(b.description)

        if col not in get_valid_moves(board):
            with output:
                clear_output(wait=True)
                print_board(board)
                print("Jogada inválida! Tenta outra.")
            return

        row = get_next_open_row(board, col)
        drop_piece(board, row, col, turn)

        # Guardar info da jogada
        last_move = col
        last_symbol = 'X' if turn == 1 else 'O'
        last_player = turn

        if root:
            root = update_root(root, board, col, turn)

        turn = 3 - turn 
        render()

        if check_winner(board, 3- turn):
            with output:
                print(f"Jogador {3-turn} ({last_symbol}) venceu!")
            game_over = True
            return
        elif not get_valid_moves(board):
            with output:
                print("Empate!")
            game_over = True
            return
        
         # alterna entre 1 e 2

        # Jogada do computador (modo pvc)
        if mode == "pvc" and turn == 2 and ai:
            col = ai(root=root, board=board)
            row = get_next_open_row(board, col)
            drop_piece(board, row, col, turn)
            last_move = col
            last_symbol = 'O'
            last_player = 2
            if root:
                root = update_root(root, board, col, turn)

            turn = 3 - turn
            render()
            if check_winner(board, 3- turn):
                with output:
                    print("Computador (O) venceu!")
                game_over = True
                return
            elif not get_valid_moves(board):
                with output:
                    print("Empate!")
                game_over = True
                return
            

    # Botões de escolha de coluna
    buttons = [widgets.Button(description=str(i)) for i in range(COLS)]
    for btn in buttons:
        btn.on_click(on_button_click)

    display(widgets.HBox(buttons))
    render()



In [ ]:
play_game_jupyter('pvc' , get_random_move)

In [ ]:
play_game_jupyter(mode= "pvp")

In [ ]:
play_game_jupyter(mode="pvc", ai= MCTS)

## C v C

In [ ]:
def simulate_game(ai1,ai2,silent=True, save=False):
    board=create_board()
    root1=None
    root2=None
    turn=1
    if callable(ai1):
        try:
            root1 = MCTSNode(board, 2)  
        except:
            pass
    if callable(ai2):
        try:
            root2 = MCTSNode(board, 1)
        except:
            pass

    
    game_data=[]
    
    while True:
       
        winner = 3 - turn if check_winner(board, 3 - turn) else 0

        if winner != 0 or not get_valid_moves(board):
            
            if winner:
                print(f"Player {winner} ({'X' if winner == 1 else 'O'}) wins!")
            else:
                print("It's a draw!")

            if save:
                save_to_csv(game_data)  # Salva os dados no CSV
            
            return winner

        if turn==1:
            col=ai1(board=board,root=root1)
            if getattr(ai1, "_is_mcts", False):
                
                game_data.append({
                'state': board_to_string(board),
                'move': col,
                })
        else:
            col=ai2(board=board,root=root2)
            if getattr(ai2, "_is_mcts", False):
                game_data.append({
                'state': board_to_string(board),
                'move': col,   
                }) 

        row=get_next_open_row(board,col)
        drop_piece(board,row,col,turn)

        if not silent:
            print_board(board)
            print(f"Computer {turn} chooses column {col}")

        if root1:
            root1=update_root(root1,board,col,turn)  
                  
        if root2:
            root2=update_root(root2,board,col,turn) 
 
        turn= 3-turn

In [ ]:
simulate_game(MCTS,get_random_move,silent=False)

## Benchmark algorithms

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def select_mcts_parameters(
    c=1.41,
    iterations=1000,
    bestchild_name="bestchild_default",
    backprograming_name="backprograming_default"
):

    bestchild_options = {
        "bestchild_default":  bestchild_default,
        "bestchild_higherVisits": bestchild_higherVisits
    }

    backprograming_options = {
        "backprograming_default": backprograming_default,
        "backprograming_greddy": backprograming_greddy
    }

   
    if bestchild_name not in bestchild_options:
        raise ValueError(f"Função bestchild '{bestchild_name}' não encontrada.")
    if backprograming_name not in backprograming_options:
        raise ValueError(f"Função backprograming '{backprograming_name}' não encontrada.")

    return {
        "c": c,
        "iterations": iterations,
        "bestchild": bestchild_options[bestchild_name],
        "backprograming": backprograming_options[backprograming_name]
    }

def benchmark(strategy1, strategy2, games=20,silent=True, name1="AI1" , name2="AI2", save=False):
    ai1_wins = 0
    draws = 0

    for i in range(games):
        print(f"Simulating game {i+1} ...")
        winner = simulate_game( ai1=strategy1, ai2=strategy2,silent=silent,save=save)
        if winner == 1:
            ai1_wins += 1
        elif winner == 0:
            draws += 1
    print(f"{name1}: {ai1_wins}, Draws: {draws}, {name2}: {games - ai1_wins - draws}")
    
def benchmark_menu_jupyter():
    output = widgets.Output()
    
    
    ai_options = ["random", "mcts"]
    ai1_dropdown = widgets.Dropdown(options=ai_options, description="AI 1:")
    ai2_dropdown = widgets.Dropdown(options=ai_options, description="AI 2:")

    
    games_slider = widgets.IntSlider(value=10, min=1, max=100, step=1, description="Games:")

    
    show_board = widgets.Checkbox(value=False, description="Show board")

    save_moves= widgets.Checkbox(value=False, description='Save moves ')
    
    start_button = widgets.Button(description="Start Benchmark", button_style="success")

    
    mcts1_box = widgets.VBox()
    mcts2_box = widgets.VBox()

    def create_mcts_config(player_label="MCTS"):
        use_default = widgets.Checkbox(value=True, description=f"{player_label}: use default")

        
        
        bestchild = widgets.Dropdown(
            options=["bestchild_default", "bestchild_higherVisits"], 
            description="Bestchild:"
        )
        backprog = widgets.Dropdown(
            options=["backprograming_default", "backprograming_greddy"], 
            description="Backprog:"
        )
        c = widgets.FloatText(value=1.41, description="c:")
        iterations = widgets.IntText(value=500, description="Iterations:")

        advanced_box = widgets.VBox([bestchild, backprog, c, iterations])
        advanced_box.layout.display = 'none'  

        def toggle_advanced_fields(change=None):
            advanced_box.layout.display = 'none' if use_default.value else 'block'

        use_default.observe(toggle_advanced_fields, names='value')

        box = widgets.VBox([use_default, advanced_box])

        def get_params():
            if use_default.value:
                return select_mcts_parameters()
            else:
                return select_mcts_parameters(
                    c=c.value,
                    iterations=iterations.value,
                    bestchild_name=bestchild.value,
                    backprograming_name=backprog.value
                )

        return box, get_params


    mcts1_cfg_box, get_mcts1_params = create_mcts_config("MCTS 1")
    mcts2_cfg_box, get_mcts2_params = create_mcts_config("MCTS 2")

    mcts1_box.children = [mcts1_cfg_box]
    mcts2_box.children = [mcts2_cfg_box]

    def toggle_mcts_boxes(*args):
        mcts1_box.layout.display = 'block' if ai1_dropdown.value == "mcts" else 'none'
        mcts2_box.layout.display = 'block' if ai2_dropdown.value == "mcts" else 'none'

    ai1_dropdown.observe(toggle_mcts_boxes, names='value')
    ai2_dropdown.observe(toggle_mcts_boxes, names='value')

    toggle_mcts_boxes()  

    def on_start_clicked(b):
        with output:
            clear_output()
            print("=== Benchmark Configuration ===")
            print(f"AI 1: {ai1_dropdown.value}")
            print(f"AI 2: {ai2_dropdown.value}")
            print(f"Games: {games_slider.value}")
            print(f"Show board: {'Yes' if show_board.value else 'No'}")
            
            
            mcts1_params = get_mcts1_params() if ai1_dropdown.value == "mcts" else {}
            mcts2_params = get_mcts2_params() if ai2_dropdown.value == "mcts" else {}

            def get_ai(name, params):
                if name == "random":
                    return get_random_move
                elif name == "mcts":
                    ai_func = lambda **kwargs: MCTS(**{**kwargs, **params})
                    ai_func._is_mcts = True  
                    return ai_func
                else:
                    raise ValueError(f"Unkonwn AI : {name}")

            benchmark(
                strategy1=get_ai(ai1_dropdown.value, mcts1_params),
                strategy2=get_ai(ai2_dropdown.value, mcts2_params),
                games=games_slider.value,
                silent=not show_board.value,
                name1=ai1_dropdown.value.upper(),
                name2=ai2_dropdown.value.upper(),
                save=save_moves

            )

    start_button.on_click(on_start_clicked)

    
    display(widgets.VBox([
        widgets.HTML(value="<h3>Benchmark Menu</h3>"),
        ai1_dropdown,
        mcts1_box,
        ai2_dropdown,
        mcts2_box,
        games_slider,
        show_board,
        save_moves,
        start_button,
        output
    ]))


In [ ]:
benchmark_menu_jupyter()